# Visualizaciones — Transmilenio XGBoost

Notebook centralizado para todas las gráficas del proyecto.

**Flujo de trabajo:**
1. Ejecutar la celda **Setup** (detecta el proyecto automáticamente)
2. Ajustar los **Parámetros** con los archivos y estaciones que quieras graficar
3. Ejecutar las secciones que necesites — cada celda es independiente

**Prerequisitos:**
- Parquets en `outputs/parquet/` → ejecutar primero `PIPELINE/03_construir_parquet.py`
- CSVs en `outputs/predicciones/` → ejecutar primero el modelo correspondiente

---

In [ ]:
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Detectar raíz del proyecto buscando config.py hacia arriba desde el directorio actual
PROYECTO_RAIZ = Path.cwd()
for _ in range(6):
    if (PROYECTO_RAIZ / "config.py").exists():
        break
    PROYECTO_RAIZ = PROYECTO_RAIZ.parent
if not (PROYECTO_RAIZ / "config.py").exists():
    raise FileNotFoundError(
        "No se encontró config.py.\n"
        "Establece la ruta manualmente descomentando la siguiente línea:\n"
        "  PROYECTO_RAIZ = Path(r'C:/tu/ruta/al/proyecto')"
    )

sys.path.insert(0, str(PROYECTO_RAIZ))
from config import RUTA_PARQUET, RUTA_PREDS, RUTA_FIGURAS

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Raíz   : {PROYECTO_RAIZ}")
print(f"Parquet: {RUTA_PARQUET}")
print(f"Preds  : {RUTA_PREDS}")
print(f"Figs   : {RUTA_FIGURAS}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  PARÁMETROS — modifica estos valores antes de ejecutar las gráficas
# ══════════════════════════════════════════════════════════════════════

# Años a incluir en las gráficas de datos históricos (Sección 1)
AÑOS_HISTORICO = [2019, 2022, 2023, 2024, 2025]

# ID de estación para las gráficas por estación individual (Sección 3)
# IDs válidos en: outputs/catalogo_estaciones.parquet → columna station_id
STATION_ID_ENTRADAS = "07107"
STATION_ID_SALIDAS  = "02000"

# Archivos CSV con predicciones del sistema completo (deben estar en outputs/predicciones/)
CSV_PRED_ENTRADAS = "predicciones_2025_prueba.csv"
CSV_PRED_SALIDAS  = "validaciones_salidas_pred_2025.csv"

# True → guardar automáticamente en outputs/FIGURAS/ al ejecutar cada celda
GUARDAR_FIGURAS = True

---
## Sección 1 — Datos históricos

Visualización de la serie temporal completa sin predicciones.
Carga los parquets directamente desde `outputs/parquet/`.

In [ ]:
# Gráfica 1 — Superposición anual: ratio (1 − Entradas/Salidas) por año
# Permite comparar visualmente el comportamiento de cada año alineado en el mismo eje de fechas.
plt.close("all")
print("Cargando datos históricos...")
data_hist = {}

for año in AÑOS_HISTORICO:
    series_e, series_s = [], []
    for fp in sorted(RUTA_PARQUET.glob(f"{año}-*-entradas.parquet")):
        df = pd.read_parquet(fp, columns=["datetime", "entradas"])
        series_e.append(df.groupby(df["datetime"].dt.date)["entradas"].sum())
    for fp in sorted(RUTA_PARQUET.glob(f"{año}-*-salidas.parquet")):
        df = pd.read_parquet(fp, columns=["datetime", "salidas"])
        series_s.append(df.groupby(df["datetime"].dt.date)["salidas"].sum())
    if not series_e or not series_s:
        print(f"  [WARN] Sin datos para {año}")
        continue
    e = pd.concat(series_e).sort_index()
    s = pd.concat(series_s).sort_index()
    e.index = pd.to_datetime(e.index).map(lambda d: d.replace(year=2000))
    s.index = pd.to_datetime(s.index).map(lambda d: d.replace(year=2000))
    data_hist[año] = (e, s)
    print(f"  {año}: {len(e)} días")

n = len(data_hist)
if n == 0:
    print("No hay datos disponibles para los años indicados en AÑOS_HISTORICO.")
else:
    fig, axes = plt.subplots(n, 1, figsize=(16, 3.5 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, (año, (e, s)) in zip(axes, data_hist.items()):
        ratio = ((1 - e / s) * 100).dropna()
        ax.plot(ratio.index, ratio.values, color="#16a34a", linewidth=0.8,
                label="(1 − Entradas/Salidas) %")
        ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", alpha=0.5)
        ax.set_ylabel(str(año), fontsize=12, fontweight="bold",
                      rotation=0, labelpad=40, va="center")
        ax.legend(loc="upper right", fontsize=9)
        ax.set_xlim(pd.Timestamp("2000-01-01"), pd.Timestamp("2000-12-31"))

    axes[0].set_title("Ratio Entradas/Salidas — Sistema Troncal Transmilenio",
                      fontsize=14, pad=12)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
    axes[-1].set_xlabel("Día del año")
    plt.tight_layout()

    if GUARDAR_FIGURAS:
        plt.savefig(RUTA_FIGURAS / "01_superposicion_anual_ratio.png", dpi=150, bbox_inches="tight")
        print("Guardada: 01_superposicion_anual_ratio.png")
    plt.show()

In [ ]:
# Gráfica 2 — Series diarias anuales: entradas y salidas absolutas por año
# Útil para detectar valores atípicos, días festivos y tendencias de recuperación post-pandemia.
plt.close("all")
print("Cargando series diarias (puede tardar unos segundos)...")

def _cargar_serie_diaria(tipo):
    archivos = sorted(glob.glob(str(RUTA_PARQUET / f"*-{tipo}.parquet")))
    partes = [pd.read_parquet(f).groupby("datetime")[tipo].sum() for f in archivos]
    return pd.concat(partes).sort_index().resample("D").sum()

entradas_dia = _cargar_serie_diaria("entradas")
salidas_dia  = _cargar_serie_diaria("salidas")
años_presentes = sorted(entradas_dia.index.year.unique())

fig, axes = plt.subplots(len(años_presentes), 1,
                          figsize=(20, 3 * len(años_presentes)), sharex=False)
fig.suptitle("Sistema Troncal Transmilenio — Entradas y Salidas diarias por año",
             fontsize=14, y=1.01)
if len(años_presentes) == 1:
    axes = [axes]
fmt_m = mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M")

for ax, año in zip(axes, años_presentes):
    e = entradas_dia[entradas_dia.index.year == año]
    s = salidas_dia[salidas_dia.index.year == año]
    ax.plot(e.index, e.values, color="#2563eb", linewidth=0.9, label="Entradas")
    ax.plot(s.index, s.values, color="#dc2626", linewidth=0.9, label="Salidas")
    ax.set_xlim(e.index.min(), e.index.max())
    ax.yaxis.set_major_formatter(fmt_m)
    ax.set_ylabel(str(año), fontsize=11, fontweight="bold",
                  rotation=0, labelpad=35, va="center")
    ax.legend(loc="upper right", fontsize=8)
    ax.tick_params(axis="x", labelsize=8)

plt.tight_layout()
if GUARDAR_FIGURAS:
    plt.savefig(RUTA_FIGURAS / "02_series_diarias_anuales.png", dpi=150, bbox_inches="tight")
    print("Guardada: 02_series_diarias_anuales.png")
plt.show()

---
## Sección 2 — Predicciones sistema completo

Comparación de predicciones XGBoost contra datos reales a nivel del sistema agregado.

**Entrada requerida:** CSVs generados por los scripts en `CODIGO/MODELO/SISTEMA/`

In [ ]:
# Gráficas 3 y 4 — Predicciones de entradas: sistema completo
# Graf 3: comparación simple (serie temporal completa)
# Graf 4: análisis 2×2 con métricas MAE/RMSE/MAPE (solo si hay datos reales disponibles)
plt.close("all")

RUTA_CSV_E = RUTA_PREDS / CSV_PRED_ENTRADAS
if not RUTA_CSV_E.exists():
    print(f"Archivo no encontrado: {RUTA_CSV_E}")
    print("Ejecuta primero: CODIGO/MODELO/SISTEMA/xgboost_transmilenio_v1.py")
else:
    pred_e = (pd.read_csv(RUTA_CSV_E, parse_dates=["datetime"])
              .sort_values("datetime").reset_index(drop=True))
    pred_e["prediccion"] = pd.to_numeric(pred_e["prediccion"], errors="coerce").clip(lower=0)
    INICIO_E, FIN_E = pred_e["datetime"].min(), pred_e["datetime"].max()

    print(f"Cargando datos reales ({INICIO_E.date()} → {FIN_E.date()})...")
    partes = []
    for f in sorted(glob.glob(str(RUTA_PARQUET / "*-entradas.parquet"))):
        df_m = pd.read_parquet(f)
        df_m["datetime"] = pd.to_datetime(df_m["datetime"])
        df_m = df_m[(df_m["datetime"] >= INICIO_E) & (df_m["datetime"] <= FIN_E)]
        if not df_m.empty:
            partes.append(df_m.groupby("datetime")["entradas"].sum())

    hay_real_e = len(partes) > 0
    df_e = None
    if hay_real_e:
        real_e = pd.concat(partes).groupby("datetime").sum().reset_index()
        real_e.columns = ["datetime", "entradas"]
        df_e = (real_e.merge(pred_e, on="datetime", how="inner")
                .sort_values("datetime").reset_index(drop=True))
        hay_real_e = len(df_e) > 0

    # ── Figura 3: comparación simple ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(20, 5))
    ax.plot(pred_e["datetime"], pred_e["prediccion"],
            color="#dc2626", lw=0.6, alpha=0.85, label="Predicción")
    if hay_real_e:
        ax.plot(df_e["datetime"], df_e["entradas"],
                color="#2563eb", lw=0.6, label="Real")
    ax.set_title(f"Predicción de entradas — sistema completo ({INICIO_E.date()} → {FIN_E.date()})")
    ax.set_ylabel("Entradas por intervalo de 15 min")
    ax.legend()
    plt.tight_layout()
    if GUARDAR_FIGURAS:
        plt.savefig(RUTA_FIGURAS / "03_pred_sistema_entradas.png", dpi=150, bbox_inches="tight")
        print("Guardada: 03_pred_sistema_entradas.png")
    plt.show()

    # ── Figura 4: análisis 2×2 con métricas ──────────────────────────────────────
    if not hay_real_e:
        print("Sin datos reales en el período predicho — omitiendo análisis con métricas.")
    else:
        y_real = df_e["entradas"].values
        y_pred = df_e["prediccion"].values
        mae  = mean_absolute_error(y_real, y_pred)
        rmse = mean_squared_error(y_real, y_pred) ** 0.5
        mape = np.mean(np.abs((y_real - y_pred) / np.where(y_real == 0, 1, y_real))) * 100
        print(f"MAE: {mae:,.0f}  RMSE: {rmse:,.0f}  MAPE: {mape:.1f}%")

        df_e["hora"] = df_e["datetime"].dt.hour
        df_e["dow"]  = df_e["datetime"].dt.weekday
        perfil_h = df_e.groupby("hora")[["entradas", "prediccion"]].mean()
        perfil_d = df_e.groupby("dow")[["entradas", "prediccion"]].mean()
        mask_2s  = df_e["datetime"] < (df_e["datetime"].min() + pd.Timedelta(weeks=2))

        COLOR_REAL = "#2563eb"
        COLOR_PRED = "#dc2626"

        fig, axes = plt.subplots(2, 2, figsize=(20, 11))
        fig.suptitle(
            f"XGBoost Entradas — MAE {mae:,.0f}  RMSE {rmse:,.0f}  MAPE {mape:.1f}%",
            fontsize=13, y=1.01,
        )

        ax = axes[0, 0]
        ax.plot(df_e["datetime"], y_real, color=COLOR_REAL, lw=0.6, label="Real")
        ax.plot(df_e["datetime"], y_pred, color=COLOR_PRED, lw=0.6, alpha=0.8, label="Predicción")
        ax.set_title("Serie completa")
        ax.set_ylabel("Entradas / 15 min")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.legend()

        ax = axes[0, 1]
        sub = df_e[mask_2s]
        ax.plot(sub["datetime"], sub["entradas"],   color=COLOR_REAL, lw=1.2, label="Real")
        ax.plot(sub["datetime"], sub["prediccion"], color=COLOR_PRED, lw=1.2, alpha=0.8, label="Predicción")
        ax.set_title("Zoom: primeras 2 semanas")
        ax.set_ylabel("Entradas / 15 min")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        ax.legend()

        ax = axes[1, 0]
        ax.plot(perfil_h.index, perfil_h["entradas"],
                color=COLOR_REAL, lw=2, marker="o", ms=4, label="Real")
        ax.plot(perfil_h.index, perfil_h["prediccion"],
                color=COLOR_PRED, lw=2, marker="o", ms=4, label="Predicción")
        ax.fill_between(perfil_h.index, perfil_h["entradas"], perfil_h["prediccion"],
                        alpha=0.12, color=COLOR_PRED)
        ax.set_title("Perfil horario promedio")
        ax.set_xlabel("Hora del día")
        ax.set_ylabel("Entradas promedio")
        ax.set_xticks(range(0, 24, 2))
        ax.legend()

        dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
        ax = axes[1, 1]
        x, w = np.arange(7), 0.35
        ax.bar(x - w/2, perfil_d["entradas"],   width=w, color=COLOR_REAL, alpha=0.85, label="Real")
        ax.bar(x + w/2, perfil_d["prediccion"], width=w, color=COLOR_PRED, alpha=0.75, label="Predicción")
        ax.set_title("Promedio por día de semana")
        ax.set_xticks(x)
        ax.set_xticklabels(dias)
        ax.set_ylabel("Entradas promedio / intervalo")
        ax.legend()

        plt.tight_layout()
        if GUARDAR_FIGURAS:
            plt.savefig(RUTA_FIGURAS / "04_analisis_entradas_2x2.png", dpi=150, bbox_inches="tight")
            print("Guardada: 04_analisis_entradas_2x2.png")
        plt.show()

In [ ]:
# Gráfica 5 — Predicciones de salidas: sistema completo
plt.close("all")

RUTA_CSV_S = RUTA_PREDS / CSV_PRED_SALIDAS
if not RUTA_CSV_S.exists():
    print(f"Archivo no encontrado: {RUTA_CSV_S}")
    print("Ejecuta primero: CODIGO/MODELO/SISTEMA/xgboost_salidas_gen.py")
else:
    pred_s = (pd.read_csv(RUTA_CSV_S, parse_dates=["datetime"])
              .sort_values("datetime").reset_index(drop=True))
    pred_s["prediccion"] = pd.to_numeric(pred_s["prediccion"], errors="coerce").clip(lower=0)
    INICIO_S, FIN_S = pred_s["datetime"].min(), pred_s["datetime"].max()

    print(f"Cargando datos reales ({INICIO_S.date()} → {FIN_S.date()})...")
    partes = []
    for f in sorted(glob.glob(str(RUTA_PARQUET / "*-salidas.parquet"))):
        df_m = pd.read_parquet(f)
        df_m["datetime"] = pd.to_datetime(df_m["datetime"])
        df_m = df_m[(df_m["datetime"] >= INICIO_S) & (df_m["datetime"] <= FIN_S)]
        if not df_m.empty:
            partes.append(df_m.groupby("datetime")["salidas"].sum())

    titulo_extra = ""
    df_s = None
    if partes:
        real_s = pd.concat(partes).groupby("datetime").sum().reset_index()
        real_s.columns = ["datetime", "salidas"]
        df_s = (real_s.merge(pred_s, on="datetime", how="inner")
                .sort_values("datetime").reset_index(drop=True))
        if len(df_s) > 0:
            mae_s  = mean_absolute_error(df_s["salidas"], df_s["prediccion"])
            rmse_s = mean_squared_error(df_s["salidas"], df_s["prediccion"]) ** 0.5
            titulo_extra = f"   MAE={mae_s:,.0f}  RMSE={rmse_s:,.0f}"
            print(f"MAE: {mae_s:,.0f}  RMSE: {rmse_s:,.0f}")

    fig, ax = plt.subplots(figsize=(20, 5))
    ax.plot(pred_s["datetime"], pred_s["prediccion"],
            color="#dc2626", lw=0.6, alpha=0.85, label="Predicción")
    if df_s is not None and len(df_s) > 0:
        ax.plot(df_s["datetime"], df_s["salidas"],
                color="#2563eb", lw=0.6, label="Real")
    ax.set_title(
        f"Predicción de salidas — sistema completo ({INICIO_S.date()} → {FIN_S.date()}){titulo_extra}"
    )
    ax.set_ylabel("Salidas por intervalo de 15 min")
    ax.legend()
    plt.tight_layout()
    if GUARDAR_FIGURAS:
        plt.savefig(RUTA_FIGURAS / "05_pred_sistema_salidas.png", dpi=150, bbox_inches="tight")
        print("Guardada: 05_pred_sistema_salidas.png")
    plt.show()

---
## Sección 3 — Análisis por estación individual

Compara la predicción XGBoost contra datos reales para una estación específica.
Ajusta `STATION_ID_ENTRADAS` y `STATION_ID_SALIDAS` en la celda de Parámetros.

**Entrada requerida:** CSVs generados por los scripts en `CODIGO/MODELO/ESTACION/`

In [ ]:
# Gráfica 6 — Predicción por estación: entradas
# Incluye serie completa, zoom 2 semanas y perfil horario (si hay datos reales).
plt.close("all")

RUTA_PRED_EST_E = RUTA_PREDS / f"validaciones_entrada_{STATION_ID_ENTRADAS}_pred_2025.csv"
if not RUTA_PRED_EST_E.exists():
    print(f"Archivo no encontrado: {RUTA_PRED_EST_E}")
    print(f"Ejecuta primero el modelo con STATION_ID = '{STATION_ID_ENTRADAS}'")
    print("Script: CODIGO/MODELO/ESTACION/xgboost_entradas_estacion.py")
else:
    pred_est_e = (pd.read_csv(RUTA_PRED_EST_E, parse_dates=["datetime"])
                  .sort_values("datetime").reset_index(drop=True))
    INICIO_EST_E = pred_est_e["datetime"].min()
    FIN_EST_E    = pred_est_e["datetime"].max()

    print(f"Cargando datos reales para estación {STATION_ID_ENTRADAS}...")
    partes_e = []
    for archivo in sorted(glob.glob(str(RUTA_PARQUET / "*-entradas.parquet"))):
        df_m = pd.read_parquet(archivo)
        df_est = df_m[df_m["station_id"] == STATION_ID_ENTRADAS]
        if not df_est.empty:
            partes_e.append(df_est[["datetime", "entradas"]])

    if not partes_e:
        raise ValueError(f"La estación '{STATION_ID_ENTRADAS}' no aparece en los parquets.")

    real_est_e = (pd.concat(partes_e, ignore_index=True)
                  .assign(datetime=lambda d: pd.to_datetime(d["datetime"]))
                  .groupby("datetime", as_index=False)["entradas"].sum()
                  .sort_values("datetime").reset_index(drop=True))

    real_pred_e = real_est_e[
        (real_est_e["datetime"] >= INICIO_EST_E) &
        (real_est_e["datetime"] <= FIN_EST_E)
    ].reset_index(drop=True)

    comp_e = None
    mae_e = rmse_e = mape_e = None
    if len(real_pred_e) > 0:
        comp_e = pred_est_e.merge(real_pred_e, on="datetime", how="inner")
        if len(comp_e) > 0:
            mae_e  = mean_absolute_error(comp_e["entradas"], comp_e["prediccion"])
            rmse_e = mean_squared_error(comp_e["entradas"], comp_e["prediccion"]) ** 0.5
            mape_e = (np.abs(comp_e["entradas"] - comp_e["prediccion"]) /
                      comp_e["entradas"].replace(0, np.nan)).mean() * 100
            print(f"MAE: {mae_e:,.1f}  RMSE: {rmse_e:,.1f}  MAPE: {mape_e:.1f}%")

    COLOR_PRED = "#fdc0cc"
    COLOR_REAL = "#7aa9ce"
    COLOR_FDS  = "#feefc4"
    COLOR_ACC  = "#d5b9e4"

    n_p = 3 if (comp_e is not None and len(comp_e) > 0) else 2
    fig, axes = plt.subplots(n_p, 1, figsize=(22, 6 * n_p))
    titulo = f"Estación {STATION_ID_ENTRADAS}  |  Predicción de entradas (XGBoost)"
    if mae_e is not None:
        titulo += f"\nMAE: {mae_e:,.1f}   RMSE: {rmse_e:,.1f}   MAPE: {mape_e:.1f}%"
    fig.suptitle(titulo, fontsize=14, fontweight="bold")

    ax = axes[0]
    ax.plot(pred_est_e["datetime"], pred_est_e["prediccion"],
            color=COLOR_PRED, lw=1.4, label="Predicción")
    if comp_e is not None:
        ax.plot(comp_e["datetime"], comp_e["entradas"],
                color=COLOR_REAL, lw=1.4, alpha=0.8, label="Real")
    ax.set_title("Serie completa — período predicho")
    ax.set_ylabel("Entradas (intervalos 15 min)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.2, color=COLOR_ACC)

    corte_2s = INICIO_EST_E + pd.Timedelta(weeks=2)
    mask_2s  = pred_est_e["datetime"] < corte_2s
    ax = axes[1]
    ax.plot(pred_est_e.loc[mask_2s, "datetime"], pred_est_e.loc[mask_2s, "prediccion"],
            color=COLOR_PRED, lw=2, label="Predicción")
    if comp_e is not None:
        mask_2s_r = comp_e["datetime"] < corte_2s
        ax.plot(comp_e.loc[mask_2s_r, "datetime"], comp_e.loc[mask_2s_r, "entradas"],
                color=COLOR_REAL, lw=2, alpha=0.85, label="Real")
    for fecha in pd.date_range(INICIO_EST_E, corte_2s, freq="D"):
        if fecha.weekday() >= 5:
            ax.axvspan(fecha, fecha + pd.Timedelta(days=1),
                       alpha=0.15, color=COLOR_FDS, zorder=0)
    ax.set_title("Zoom: primeras 2 semanas (fines de semana sombreados)")
    ax.set_ylabel("Entradas (intervalos 15 min)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.2, color=COLOR_ACC)

    if n_p == 3 and comp_e is not None:
        perfil_h_pred = pred_est_e.groupby(pred_est_e["datetime"].dt.hour)["prediccion"].mean()
        perfil_h_real = comp_e.groupby(comp_e["datetime"].dt.hour)["entradas"].mean()
        ax = axes[2]
        ax.plot(perfil_h_pred.index, perfil_h_pred.values,
                color=COLOR_PRED, lw=2, marker="o", ms=3, label="Predicción")
        ax.plot(perfil_h_real.index, perfil_h_real.values,
                color=COLOR_REAL, lw=2, marker="s", ms=3, label="Real")
        ax.set_title("Perfil horario promedio")
        ax.set_xlabel("Hora del día")
        ax.set_ylabel("Entradas promedio")
        ax.set_xticks(range(0, 24, 2))
        ax.legend()
        ax.grid(True, alpha=0.2, color=COLOR_ACC)

    fig.subplots_adjust(top=0.93, hspace=0.35)
    if GUARDAR_FIGURAS:
        salida = RUTA_FIGURAS / f"06_estacion_entradas_{STATION_ID_ENTRADAS}.png"
        plt.savefig(salida, dpi=150, bbox_inches="tight")
        print(f"Guardada: {salida.name}")
    plt.show()

In [ ]:
# Gráfica 7 — Predicción por estación: salidas
# Misma estructura que la celda anterior pero para flujos de salida.
plt.close("all")

RUTA_PRED_EST_S = RUTA_PREDS / f"validaciones_salida_{STATION_ID_SALIDAS}_pred_2025.csv"
if not RUTA_PRED_EST_S.exists():
    print(f"Archivo no encontrado: {RUTA_PRED_EST_S}")
    print(f"Ejecuta primero el modelo con STATION_ID = '{STATION_ID_SALIDAS}'")
    print("Script: CODIGO/MODELO/ESTACION/xgboost_salidas_estacion.py")
else:
    pred_est_s = (pd.read_csv(RUTA_PRED_EST_S, parse_dates=["datetime"])
                  .sort_values("datetime").reset_index(drop=True))
    INICIO_EST_S = pred_est_s["datetime"].min()
    FIN_EST_S    = pred_est_s["datetime"].max()

    print(f"Cargando datos reales para estación {STATION_ID_SALIDAS}...")
    partes_s = []
    for archivo in sorted(glob.glob(str(RUTA_PARQUET / "*-salidas.parquet"))):
        df_m = pd.read_parquet(archivo)
        df_est = df_m[df_m["station_id"] == STATION_ID_SALIDAS]
        if not df_est.empty:
            partes_s.append(df_est[["datetime", "salidas"]])

    if not partes_s:
        raise ValueError(f"La estación '{STATION_ID_SALIDAS}' no aparece en los parquets.")

    real_est_s = (pd.concat(partes_s, ignore_index=True)
                  .assign(datetime=lambda d: pd.to_datetime(d["datetime"]))
                  .groupby("datetime", as_index=False)["salidas"].sum()
                  .sort_values("datetime").reset_index(drop=True))

    real_pred_s = real_est_s[
        (real_est_s["datetime"] >= INICIO_EST_S) &
        (real_est_s["datetime"] <= FIN_EST_S)
    ].reset_index(drop=True)

    comp_s = None
    mae_s = rmse_s = mape_s = None
    if len(real_pred_s) > 0:
        comp_s = pred_est_s.merge(real_pred_s, on="datetime", how="inner")
        if len(comp_s) > 0:
            mae_s  = mean_absolute_error(comp_s["salidas"], comp_s["prediccion"])
            rmse_s = mean_squared_error(comp_s["salidas"], comp_s["prediccion"]) ** 0.5
            mape_s = (np.abs(comp_s["salidas"] - comp_s["prediccion"]) /
                      comp_s["salidas"].replace(0, np.nan)).mean() * 100
            print(f"MAE: {mae_s:,.1f}  RMSE: {rmse_s:,.1f}  MAPE: {mape_s:.1f}%")

    COLOR_PRED = "#fdc0cc"
    COLOR_REAL = "#7aa9ce"
    COLOR_FDS  = "#feefc4"
    COLOR_ACC  = "#d5b9e4"

    n_p = 3 if (comp_s is not None and len(comp_s) > 0) else 2
    fig, axes = plt.subplots(n_p, 1, figsize=(22, 6 * n_p))
    titulo = f"Estación {STATION_ID_SALIDAS}  |  Predicción de salidas (XGBoost)"
    if mae_s is not None:
        titulo += f"\nMAE: {mae_s:,.1f}   RMSE: {rmse_s:,.1f}   MAPE: {mape_s:.1f}%"
    fig.suptitle(titulo, fontsize=14, fontweight="bold")

    ax = axes[0]
    ax.plot(pred_est_s["datetime"], pred_est_s["prediccion"],
            color=COLOR_PRED, lw=1.4, label="Predicción")
    if comp_s is not None:
        ax.plot(comp_s["datetime"], comp_s["salidas"],
                color=COLOR_REAL, lw=1.4, alpha=0.8, label="Real")
    ax.set_title("Serie completa — período predicho")
    ax.set_ylabel("Salidas (intervalos 15 min)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.2, color=COLOR_ACC)

    corte_2s = INICIO_EST_S + pd.Timedelta(weeks=2)
    mask_2s  = pred_est_s["datetime"] < corte_2s
    ax = axes[1]
    ax.plot(pred_est_s.loc[mask_2s, "datetime"], pred_est_s.loc[mask_2s, "prediccion"],
            color=COLOR_PRED, lw=2, label="Predicción")
    if comp_s is not None:
        mask_2s_r = comp_s["datetime"] < corte_2s
        ax.plot(comp_s.loc[mask_2s_r, "datetime"], comp_s.loc[mask_2s_r, "salidas"],
                color=COLOR_REAL, lw=2, alpha=0.85, label="Real")
    for fecha in pd.date_range(INICIO_EST_S, corte_2s, freq="D"):
        if fecha.weekday() >= 5:
            ax.axvspan(fecha, fecha + pd.Timedelta(days=1),
                       alpha=0.15, color=COLOR_FDS, zorder=0)
    ax.set_title("Zoom: primeras 2 semanas (fines de semana sombreados)")
    ax.set_ylabel("Salidas (intervalos 15 min)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.2, color=COLOR_ACC)

    if n_p == 3 and comp_s is not None:
        perfil_h_pred = pred_est_s.groupby(pred_est_s["datetime"].dt.hour)["prediccion"].mean()
        perfil_h_real = comp_s.groupby(comp_s["datetime"].dt.hour)["salidas"].mean()
        ax = axes[2]
        ax.plot(perfil_h_pred.index, perfil_h_pred.values,
                color=COLOR_PRED, lw=2, marker="o", ms=3, label="Predicción")
        ax.plot(perfil_h_real.index, perfil_h_real.values,
                color=COLOR_REAL, lw=2, marker="s", ms=3, label="Real")
        ax.set_title("Perfil horario promedio")
        ax.set_xlabel("Hora del día")
        ax.set_ylabel("Salidas promedio")
        ax.set_xticks(range(0, 24, 2))
        ax.legend()
        ax.grid(True, alpha=0.2, color=COLOR_ACC)

    fig.subplots_adjust(top=0.93, hspace=0.35)
    if GUARDAR_FIGURAS:
        salida = RUTA_FIGURAS / f"07_estacion_salidas_{STATION_ID_SALIDAS}.png"
        plt.savefig(salida, dpi=150, bbox_inches="tight")
        print(f"Guardada: {salida.name}")
    plt.show()